In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from contextlib import suppress
from types import SimpleNamespace
from pathlib import Path

from tqdm import tqdm, trange
from utils import io_utils, ml_utils

In [ ]:
cp = SimpleNamespace()
cp.SAVE_DIR = "./data_figs/"
cp.ANIMAL = "mahler"  # amadeus | mahler
cp.SEQ = 2
cp.GENERALIZATION_SEQ = 2
cp.EXPERIMENT_FILE = Path(f"/media/naeem_md93/DRIVE/MNAV/Fig1/{cp.ANIMAL}_exp.mat")
cp.FILE_PATH = Path(f"/media/naeem_md93/DRIVE/MNAV/Fig1/123fig1ce_data_{cp.ANIMAL}_seq{cp.SEQ}.mat")
cp.MTT_DIR = Path(f"/media/naeem_md93/DRIVE/MNAV/EC/")

# MNAV

In [ ]:
if not cp.FILE_PATH.exists():
    assert cp.EXPERIMENT_FILE.exists(), f"{str(cp.EXPERIMENT_FILE)} must exist when {str(cp.FILE_PATH)} is not available"
    exp_data = io_utils.load_mat_file(file_path=cp.EXPERIMENT_FILE)

    if cp.ANIMAL == "amadeus" and cp.SEQ == cp.GENERALIZATION_SEQ:
        expid = (
            exp_data[f"a_expid_seq{cp.SEQ}"].tolist() +
            exp_data["a_expid_seq12"].tolist() +
            exp_data["a_expid_seq4"].tolist()
        )
    elif cp.ANIMAL == "mahler" and cp.SEQ == cp.GENERALIZATION_SEQ:
        expid = (
            exp_data[f"m_expid_seq{cp.SEQ}"].tolist() +
            exp_data["m_expid_seq12"].tolist() +
            exp_data["m_expid_seq4"].tolist() +
            exp_data["m_expid_seq124"].tolist()
        )

    expallpairs = exp_data[f"{cp.ANIMAL[0]}_expid_seq12"]

    allpairsstart = 0

    regreslope = []
    regres_adjrsq = []
    vregreslope = []
    seen, unseen, allpairs = [], [], []
    mtr, vtr, goodsession, vgoodsession = [], [], [], []

    t = tqdm(enumerate(expid))
    for exp_idx, exp in t:
        t.set_description_str(f"{exp}")

        if (cp.MTT_DIR / f"{exp}.mwk" / f"{exp}.mat").exists():
            exp_file_path = cp.MTT_DIR / f"{exp}.mwk" / f"{exp}.mat"
        elif (cp.MTT_DIR / f"{exp}.mwk" / f"concat_{exp[:-2]}.mat").exists():
            exp_file_path = cp.MTT_DIR / f"{exp}.mwk" / f"concat_{exp[:-2]}.mat"
        else:
            print(f"{exp} not found. skipping")
            continue

        mat_data = io_utils.load_mat_file(exp_file_path)

        if exp == expallpairs[0]:
            allpairsstart = 1

        if allpairsstart == 0:
            seqq = mat_data["seqq"] * np.ones(shape=mat_data["tp"].shape)  # What is this???

        g_mask = np.where(
            (mat_data["mask"] == 1)
            & (np.abs(mat_data["tp"]) < 10)
            & (mat_data["seqq"] == cp.SEQ)
        )[0]

        temp_1 = ml_utils.fitlm(np.abs(mat_data["ta"][g_mask]), np.abs(mat_data["tp"][g_mask]))
        temp_2 = ml_utils.fitlm(mat_data["ta"][g_mask], mat_data["tp"][g_mask])
        regreslope.append(np.array([
            temp_1.Coefficients.Estimate[1],
            temp_2.Coefficients.Estimate[1]
        ]))
        regres_adjrsq.append(temp_2.Rsquared.Adjusted)


        g_mask = np.where(
            (mat_data["mask"] < 1)
            & (np.abs(mat_data["tp"]) < 10)
            & (mat_data["seqq"] == cp.SEQ)
        )
        temp_1 = ml_utils.fitlm(np.abs(mat_data["ta"][g_mask]), np.abs(mat_data["tp"][g_mask]))
        temp_2 = ml_utils.fitlm(mat_data["ta"][g_mask], mat_data["tp"][g_mask])
        vregreslope.append(np.array([
            temp_1.Coefficients.Estimate[1],
            temp_2.Coefficients.Estimate[1]
        ]))

        seen.append(
            np.any(np.where(mat_data["ta"] == 3.25)[0])
            & (len(np.unique(mat_data["ta"])) < 10)
        )
        unseen.append(
            np.any(np.where(mat_data["ta"] == -3.25)[0])
        )
        allpairs.append(len(np.unique(mat_data["ta"]) == 10))

        mtr.append(np.where((mat_data["mask"] == 1) & (mat_data["seqq"] == cp.SEQ))[0].sum())
        vtr.append(np.where((mat_data["mask"] < 1) & (mat_data["seqq"] == cp.SEQ))[0].sum())

        goodsession.append((
            np.where()
        ))

In [ ]:
 data = io_utils.load_mat_file(cp.FILE_PATH)
print(data.keys())

In [ ]:
mnavtr = np.where(data["regreslope"][:, 1] != 0)[0][0]
switchtr = np.where(data["seen"])[0][0]

temp = np.where(data["allpairs"] == 1)[0]
temp = temp[temp > switchtr]
switchallpairs = temp[0]

In [ ]:
plt.figure(figsize=(8, 8))


sns.lineplot(
    data["regreslope"][mnavtr:switchtr - 1, 1],
    color="b"
)


plt.show()

In [ ]:
data["regreslope"][mnavtr:switchtr - 1, :].shape